# BGK PLN — dashboard (extended)

Sekcje:

1. **Outstanding per program** — running total (mld PLN), z `v_bgk_outstanding_timeline`.
2. **Outstanding per coupon kind** — fixed vs floater w czasie, stacked.
3. **Maturity ladder** — outstanding per maturity year × program (stacked bar).
4. **Funding pace YTD** — kumulatywna emisja w czasie roku, year-over-year.
5. **Spread vs POLGB** — dual-panel (fixed: yield-space, floater: DM-space z coupon margin z XLSX).
6. **FPC auction B/C + NK** — z `v_bgk_auction_metrics` (PDF data, FPC only).
7. **Recent issuances** — ostatnie 20 emisji cross-program.

Źródło: shared Supabase z CETO_DOWNLOADER.

**Setup:** `pip install -r requirements-notebook.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org`, ustaw `SUPABASE_URL` + `SUPABASE_SERVICE_ROLE_KEY` w `.env`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
from IPython.display import display
from plotly.subplots import make_subplots

try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

pio.renderers.default = 'notebook_connected'

SUPABASE_URL = os.environ['SUPABASE_URL'].rstrip('/')
for suffix in ('/rest/v1', '/rest'):
    if SUPABASE_URL.endswith(suffix):
        SUPABASE_URL = SUPABASE_URL[: -len(suffix)]
        break
SUPABASE_KEY = os.environ['SUPABASE_SERVICE_ROLE_KEY']

HEADERS = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
}
PAGE_SIZE = 1000


def fetch_view(name: str, query: str = '?select=*') -> pd.DataFrame:
    rows: list = []
    for page in range(200):
        offset = page * PAGE_SIZE
        h = {**HEADERS, 'Range-Unit': 'items',
             'Range': f'{offset}-{offset + PAGE_SIZE - 1}'}
        r = requests.get(f'{SUPABASE_URL}/rest/v1/{name}{query}', headers=h, timeout=60)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            break
    return pd.DataFrame(rows)


print(f'Connected to {SUPABASE_URL}')

## 1. Outstanding PLN per program

Step chart per program (FPC / KFD / FP / FWSZ / własne). Z `v_bgk_outstanding_timeline`.

In [ ]:
df_timeline = fetch_view(
    'v_bgk_outstanding_timeline',
    '?currency=eq.PLN&select=*&order=event_date.asc',
)
df_timeline['event_date'] = pd.to_datetime(df_timeline['event_date'])
df_timeline['outstanding_bn'] = df_timeline['running_outstanding'].astype(float) / 1e9

program_order = (df_timeline.groupby('program')['outstanding_bn'].last()
                 .sort_values(ascending=False).index.tolist())

fig = go.Figure()
for prog in program_order:
    sub = df_timeline[df_timeline['program'] == prog].sort_values('event_date')
    fig.add_trace(go.Scatter(
        x=sub['event_date'], y=sub['outstanding_bn'],
        mode='lines', line_shape='hv', name=prog,
        hovertemplate=f'{prog}<br>%{{x|%Y-%m-%d}}<br>%{{y:.2f}} mld PLN<extra></extra>',
    ))
fig.update_layout(
    title='BGK PLN outstanding per program (mld PLN)',
    yaxis_title='Outstanding (mld PLN)', hovermode='x unified',
    height=420, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

latest = (df_timeline.sort_values('event_date').groupby('program')
          .agg(outstanding_bn=('outstanding_bn', 'last'),
               last_event=('event_date', 'last'))
          .sort_values('outstanding_bn', ascending=False))
display(latest.style.format({'outstanding_bn': '{:.2f} mld PLN'}))

## 2. Outstanding per coupon kind

Fixed (stałe) vs floater (zmienne) split outstanding w czasie. Stacked area mld PLN. Liczone Python-side z `bgk_auctions` (issue +, maturity −, cumsum per kind).

In [ ]:
df_a = fetch_view(
    'bgk_auctions',
    '?currency=eq.PLN&select=isin,series,program,coupon_kind,issue_date,maturity_date,issue_amount',
)
df_a['issue_date'] = pd.to_datetime(df_a['issue_date'])
df_a['maturity_date'] = pd.to_datetime(df_a['maturity_date'])
df_a['issue_amount'] = pd.to_numeric(df_a['issue_amount'])
df_a['coupon_kind'] = df_a['coupon_kind'].fillna('unknown')

issuances = df_a[['issue_date', 'coupon_kind', 'issue_amount']].rename(
    columns={'issue_date': 'event_date'})
redemptions = df_a[['maturity_date', 'coupon_kind', 'issue_amount']].rename(
    columns={'maturity_date': 'event_date'})
redemptions = redemptions.assign(issue_amount=lambda x: -x['issue_amount'])

events = pd.concat([issuances, redemptions]).sort_values('event_date')
events['running'] = events.groupby('coupon_kind')['issue_amount'].cumsum()
events['running_bn'] = events['running'] / 1e9

fig = go.Figure()
for kind in sorted(events['coupon_kind'].unique()):
    sub = events[events['coupon_kind'] == kind].sort_values('event_date')
    fig.add_trace(go.Scatter(
        x=sub['event_date'], y=sub['running_bn'],
        mode='lines', line_shape='hv', name=kind, stackgroup='one',
        hovertemplate=f'{kind}<br>%{{x|%Y-%m-%d}}<br>%{{y:.2f}} mld PLN<extra></extra>',
    ))
fig.update_layout(
    title='BGK PLN outstanding per coupon kind (mld PLN, stacked)',
    yaxis_title='Outstanding (mld PLN)', hovermode='x unified',
    height=420, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

# Snapshot today
now = pd.Timestamp.today()
active = df_a[(df_a['issue_date'] <= now) & (df_a['maturity_date'] > now)]
mix = (active.groupby('coupon_kind')['issue_amount'].sum() / 1e9).sort_values(ascending=False)
mix_pct = (mix / mix.sum() * 100).round(1)
print(f'\nActive outstanding split (today, {now.date()}):')
for k, v in mix.items():
    print(f'  {k:<10} {v:8.2f} mld PLN  ({mix_pct[k]:5.1f}%)')

## 3. Maturity ladder

Per maturity year × program, stacked bar. Pokazuje kiedy bond-y dochodzą do wykupu — refinansowanie pressure timing. Tylko obligacje active na dziś.

In [ ]:
now = pd.Timestamp.today()
active_pln = df_a[(df_a['issue_date'] <= now) & (df_a['maturity_date'] > now)].copy()
active_pln['mat_year'] = active_pln['maturity_date'].dt.year
active_pln['issue_amount_bn'] = active_pln['issue_amount'] / 1e9

ladder = (active_pln.groupby(['mat_year', 'program'])['issue_amount_bn']
          .sum().reset_index().sort_values('mat_year'))

# Order programs by total size in ladder for consistent stack ordering
prog_order_ladder = (ladder.groupby('program')['issue_amount_bn'].sum()
                     .sort_values(ascending=False).index.tolist())

fig = go.Figure()
for prog in prog_order_ladder:
    sub = ladder[ladder['program'] == prog]
    fig.add_trace(go.Bar(
        x=sub['mat_year'], y=sub['issue_amount_bn'], name=prog,
        hovertemplate=f'{prog}<br>mat: %{{x}}<br>%{{y:.2f}} mld PLN<extra></extra>',
    ))

fig.update_layout(
    title='BGK PLN maturity ladder (mld PLN, stacked per program)',
    barmode='stack',
    xaxis_title='Maturity year', yaxis_title='Outstanding (mld PLN)',
    template='plotly_white', height=420, hovermode='x unified',
)
fig.show()

ladder_total = ladder.groupby('mat_year')['issue_amount_bn'].sum().sort_index()
print(f'\nTotal active outstanding: {ladder_total.sum():.2f} mld PLN')
print(f'Peak refi year: {ladder_total.idxmax()} ({ladder_total.max():.2f} mld PLN)')
print(f'Next 3 years refi: {ladder_total.loc[now.year:now.year+2].sum():.2f} mld PLN')

## 4. Funding pace — cumulative YTD by year

Per year, kumulatywna emisja od początku roku. Pokazuje **tempo** pozyskiwania finansowania year-over-year. Wszystkie PLN programy, all coupon kinds.

In [ ]:
issuances_only = df_a[['issue_date', 'issue_amount']].copy()
issuances_only['year'] = issuances_only['issue_date'].dt.year
issuances_only['doy'] = issuances_only['issue_date'].dt.dayofyear
issuances_only = issuances_only.sort_values(['year', 'doy'])
issuances_only['cum_bn'] = issuances_only.groupby('year')['issue_amount'].cumsum() / 1e9

fig = go.Figure()
years_sorted = sorted(issuances_only['year'].unique())
for year in years_sorted:
    sub = issuances_only[issuances_only['year'] == year]
    # Use thicker line for current year
    is_current = (year == pd.Timestamp.today().year)
    fig.add_trace(go.Scatter(
        x=sub['doy'], y=sub['cum_bn'],
        mode='lines', line_shape='hv', name=str(year),
        line=dict(width=3 if is_current else 1.5),
        hovertemplate=f'{year}<br>day %{{x}}<br>%{{y:.2f}} mld PLN<extra></extra>',
    ))

fig.update_layout(
    title='BGK PLN cumulative new issuance YTD (mld PLN)',
    xaxis_title='Day of year (1 = Jan 1)', yaxis_title='Cumulative (mld PLN)',
    template='plotly_white', height=420,
    legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.02),
)
fig.show()

year_totals = issuances_only.groupby('year').agg(
    total_bn=('issue_amount', lambda x: x.sum() / 1e9),
    n_events=('issue_amount', 'count'),
).sort_index()
print('\nTotal new issuance per year (mld PLN):')
display(year_totals.style.format({'total_bn': '{:,.2f}'}))

## 5. Spread vs POLGB curve

Per-issuance spread (bp) z `v_bgk_issuance_spread`. Dwa panele:

- **Top — fixed-coupon**: yield-space (`bgk_yield - polgb_yield_interp`) × 100 bp.
- **Bottom — floaters**: DM-space (`bgk_true_DM - polgb_WZ_DM_interp`) × 100 bp; `bgk_true_DM = price-implied + coupon margin` (margin z XLSX 'Kupon').

FPC1140 (~14Y) zostaje NULL — poza WZ curve.

In [ ]:
df_spread = fetch_view(
    'v_bgk_issuance_spread',
    '?spread_bp=not.is.null&select=*&order=issue_date.asc',
)
df_spread['issue_date'] = pd.to_datetime(df_spread['issue_date'])
for c in ['spread_bp', 'tenor_years', 'bgk_yield_pct', 'bgk_true_dm_pct',
          'bgk_margin_bp', 'bgk_price_pct']:
    if c in df_spread.columns:
        df_spread[c] = pd.to_numeric(df_spread[c], errors='coerce')

fixed = df_spread[df_spread['bgk_coupon_kind'] == 'stałe'].copy()
floater = df_spread[df_spread['bgk_coupon_kind'] == 'zmienne'].copy()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        f'Fixed-coupon spread vs POLGB nominal curve (bp) — {len(fixed)} emisji',
        f'Floater spread vs POLGB WZ curve (bp, DM-space) — {len(floater)} emisji',
    ),
)

for series in sorted(fixed['series'].unique()):
    sub = fixed[fixed['series'] == series].sort_values('issue_date')
    fig.add_trace(go.Scatter(
        x=sub['issue_date'], y=sub['spread_bp'],
        mode='lines+markers', name=series, legendgroup='fixed',
        customdata=sub[['program', 'tenor_years']].values,
        hovertemplate=(f'{series} (%{{customdata[0]}})<br>'
                       '%{x|%Y-%m-%d}<br>spread: %{y:.1f} bp<br>'
                       'tenor: %{customdata[1]:.1f} y<extra></extra>'),
    ), row=1, col=1)

for series in sorted(floater['series'].unique()):
    sub = floater[floater['series'] == series].sort_values('issue_date')
    fig.add_trace(go.Scatter(
        x=sub['issue_date'], y=sub['spread_bp'],
        mode='lines+markers', name=series, legendgroup='floater',
        customdata=sub[['program', 'tenor_years', 'bgk_margin_bp']].values,
        hovertemplate=(f'{series} (%{{customdata[0]}})<br>'
                       '%{x|%Y-%m-%d}<br>spread: %{y:.1f} bp<br>'
                       'tenor: %{customdata[1]:.1f} y<br>'
                       'coupon margin: %{customdata[2]:.0f} bp<extra></extra>'),
    ), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4, row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4, row=2, col=1)
fig.update_yaxes(title_text='Spread (bp)', row=1, col=1)
fig.update_yaxes(title_text='Spread (bp)', row=2, col=1)
fig.update_layout(
    height=720, template='plotly_white', hovermode='x unified',
    legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.02),
)
fig.show()

summary = (df_spread.groupby(['bgk_coupon_kind', 'series'])['spread_bp']
           .agg(['count', 'mean', 'min', 'max']).round(1)
           .rename(columns={'count': 'n', 'mean': 'avg_bp', 'min': 'min_bp', 'max': 'max_bp'}))
print('\nSpread summary per series:')
display(summary)

## 6. FPC auction metrics — bid-to-cover + NK share

Z `v_bgk_auction_metrics` (PDF data). FPC only — pozostałe programy nie publikują wyników aukcyjnych.

- **B/C** > 1 = popyt pokrył sprzedaż. **B/C** > 2 = bardzo mocny popyt.
- **NK share of demand** = jaki procent łącznego popytu przyszedł z ofert niekonkurencyjnych. Wysoka NK = część rynku rezerwuje sobie zapas po stop price (chce dostać bond, nie chce ryzykować mis-pricing).

In [ ]:
df_metrics = fetch_view(
    'v_bgk_auction_metrics',
    '?series=like.FPC*&select=auction_date,series,demand_total_mln,sold_total_mln,'
    'bid_to_cover,nc_share_demand,nc_share_sold&order=auction_date.asc',
)
df_metrics['auction_date'] = pd.to_datetime(df_metrics['auction_date'])
for c in ['demand_total_mln', 'sold_total_mln', 'bid_to_cover',
          'nc_share_demand', 'nc_share_sold']:
    df_metrics[c] = pd.to_numeric(df_metrics[c], errors='coerce')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=('Bid-to-cover per FPC series',
                                     'NK share of demand per FPC series'))
for series in sorted(df_metrics['series'].unique()):
    sub = df_metrics[df_metrics['series'] == series].sort_values('auction_date')
    fig.add_trace(go.Scatter(
        x=sub['auction_date'], y=sub['bid_to_cover'],
        mode='lines+markers', name=series, legendgroup=series,
        hovertemplate=f'{series}<br>%{{x|%Y-%m-%d}}<br>B/C: %{{y:.2f}}<extra></extra>',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sub['auction_date'], y=sub['nc_share_demand'],
        mode='lines+markers', name=series, legendgroup=series, showlegend=False,
        hovertemplate=f'{series}<br>%{{x|%Y-%m-%d}}<br>NK: %{{y:.1%}}<extra></extra>',
    ), row=2, col=1)

fig.add_hline(y=1.0, line_dash='dash', line_color='gray', opacity=0.4, row=1, col=1)
fig.add_hline(y=2.0, line_dash='dot', line_color='lightgreen', opacity=0.5, row=1, col=1)
fig.update_yaxes(title_text='B/C ratio', row=1, col=1)
fig.update_yaxes(title_text='NK share', tickformat='.0%', row=2, col=1)
fig.update_layout(height=600, template='plotly_white', hovermode='x unified',
                   legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.02))
fig.show()

agg = (df_metrics.groupby('series')
       .agg(n_auctions=('auction_date', 'count'),
            avg_bid_to_cover=('bid_to_cover', 'mean'),
            avg_nc_share=('nc_share_demand', 'mean'),
            total_sold_mln=('sold_total_mln', 'sum'))
       .round({'avg_bid_to_cover': 2, 'avg_nc_share': 3, 'total_sold_mln': 0}))
print('\nFPC auction stats per series:')
display(agg)

## 7. Recent issuances — last 20 across all PLN programs

Per-emisja snapshot. Floatery pokazują `bgk_true_dm` (price + margin), fixed pokazują `bgk_yield`. Spread vs POLGB w ostatniej kolumnie.

In [ ]:
df_recent = fetch_view(
    'v_bgk_issuance_spread',
    '?select=issue_date,program,series,isin,tenor_years,bgk_coupon_kind,'
    'bgk_margin_bp,bgk_yield_pct,bgk_true_dm_pct,bgk_price_pct,issue_amount,spread_bp'
    '&order=issue_date.desc&limit=20',
)
for c in ['tenor_years', 'bgk_margin_bp', 'bgk_yield_pct', 'bgk_true_dm_pct',
          'bgk_price_pct', 'issue_amount', 'spread_bp']:
    df_recent[c] = pd.to_numeric(df_recent[c], errors='coerce')

df_recent['rate_pct'] = df_recent['bgk_yield_pct'].fillna(df_recent['bgk_true_dm_pct'])
df_recent['issue_amount_mln'] = df_recent['issue_amount'] / 1e6

df_show = df_recent.rename(columns={
    'issue_date': 'date',
    'bgk_coupon_kind': 'kind',
    'bgk_margin_bp': 'margin_bp',
    'bgk_price_pct': 'price',
    'tenor_years': 'tenor_y',
})[['date', 'program', 'series', 'isin', 'kind', 'tenor_y',
    'margin_bp', 'price', 'rate_pct', 'issue_amount_mln', 'spread_bp']]

display(df_show.style.format({
    'tenor_y':          '{:.2f}',
    'margin_bp':        '{:.0f}',
    'price':            '{:.3f}',
    'rate_pct':         '{:.3f}',
    'issue_amount_mln': '{:,.0f}',
    'spread_bp':        '{:.1f}',
}, na_rep='—').hide(axis='index'))